# 02. 설정 → Snapshot: `Snapshot.build` 안을 한 줄씩

| 단계 | QuantTool 코드 | ORE 원본 |
|---|---|---|
| 레시피 읽기 | `src/quanttool/config/__init__.py` `load_config`, `configs/` | `OREData/ored/configuration/curveconfigurations.hpp` |
| 시세 읽기 | `src/quanttool/market/quotes.py` `Loader`, `QuoteStore` | `OREData/ored/marketdata/loader.hpp` |
| 빌더 찾기 | `src/quanttool/core/registry.py` `BUILDERS` | (C++ 클래스 계층) |
| 순서 정하기 | `src/quanttool/core/dependency.py` | `OREData/ored/marketdata/dependencygraph.hpp` |
| 구축·보관 | `src/quanttool/market/market.py` `Snapshot.build` | `OREData/ored/marketdata/todaysmarket.hpp`, `marketimpl.hpp` |

ORE 경로는 `references/Engine/` 기준. > 데이터: `tests/fixtures/ust_par_sample.csv`는 재무부 CSV와 **형식만 같은 합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE_CSV = ROOT / "tests" / "fixtures" / "ust_par_sample.csv"  # 합성 샘플 (실제 시세 아님)

from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.config import load_config
from quanttool.core.dependency import DependencyGraph
from quanttool.core.registry import BUILDERS
from quanttool.market import QuoteStore, Snapshot, UstParCsvLoader
from quanttool.market.market import to_ql_date

ASOF = date(2026, 10, 23)

## 1. 레시피 (무엇을 어떻게 만들지)
`configs/market/curves/ust.yaml`이 "어떤 시세로, 어떤 헬퍼로, 어떤 보간으로"를 정한다. 코드에 숫자가 없다.

In [ ]:
cfg = load_config(ROOT / "configs")
print("구축 대상:", cfg.todaysmarket.yield_curves)
cfg.curves["UST"].model_dump()

In [ ]:
cfg.conventions["UST"].model_dump()

## 2. 시세 읽기 → QuoteStore
Loader는 그 날짜의 시세를 `MarketQuote(asof, id, value)` 목록으로 준다. QuoteStore는 id마다 `SimpleQuote`를 하나 만든다. **모든 구조물은 여기 Quote에 연결된다.**

In [ ]:
loader = UstParCsvLoader(SAMPLE_CSV)
quotes = loader.load(ASOF)
pd.DataFrame([q.__dict__ for q in quotes])

In [ ]:
ql.Settings.instance().evaluationDate = to_ql_date(ASOF)
store = QuoteStore(quotes)
store["UST/PAR/10Y"], store["UST/PAR/10Y"].value()

## 3. 빌더 찾기와 구축 순서
레시피의 `type`(`YieldCurve`)으로 등록부에서 빌더를 찾는다. 커브가 여러 개이고 서로를 참조하면(예: 주식 선도 → 할인 커브) 의존 그래프가 순서를 정한다. 지금은 UST 하나뿐이다.

In [ ]:
print("등록된 빌더:", BUILDERS.keys())
builder = BUILDERS["YieldCurve"](cfg)
recipe = cfg.curves["UST"]
graph = DependencyGraph({("YieldCurve", "UST"): builder.requires(recipe)})
graph.order([("YieldCurve", "UST")])

## 4. 구축하고 보관소에 등록
`_register`는 커브를 `RelinkableYieldTermStructureHandle`에 넣어 이름으로 꺼낼 수 있게 한다.

In [ ]:
snap = Snapshot(ASOF, store)
built = builder.build(recipe, store, snap)
snap._register(("YieldCurve", "UST"), built)  # Snapshot.build가 하는 일 (내부 메소드)
print("헬퍼 수", len(built.helpers), " 보관된 커브", snap.yield_curve_names())

## 5. 위 1~4단계 = 한 줄
가격결정 코드는 이렇게 만든 시장에서 **이름으로** 커브를 꺼낸다(원칙 3).

In [ ]:
snap = Snapshot.build(ASOF, cfg, loader)
curve = snap.yield_curve("UST")
d = ql.Date(17, 3, 2031)
print("DF", curve.discount(d))
print("zero", curve.zeroRate(d, ql.Actual365Fixed(), ql.Continuous).rate())
print(
    "1Y fwd from 2031-03-17",
    curve.forwardRate(d, d + ql.Period("1Y"), ql.Actual365Fixed(), ql.Continuous).rate(),
)

## 6. 진단과 커브 표
`src/quanttool/market/diagnostics.py`, `src/quanttool/analytics/market_monitor.py`

In [ ]:
from quanttool.analytics.market_monitor import curve_shape, curve_table
from quanttool.market.diagnostics import helper_repricing

helper_repricing(snap, "UST")

In [ ]:
table = curve_table(snap, "UST", cfg)
display((table[["par", "zero_cc", "fwd_1y"]] * 100).round(3))
curve_shape(table)

## 7. 날짜가 다른 시장
평가일이 전역이라, 다른 날짜의 시장은 `active()` 안에서 **번갈아** 평가한다(`docs/architecture.md` 6.4).

In [ ]:
snap_prev = Snapshot.build(date(2026, 10, 22), cfg, loader)
for s in (snap_prev, snap):
    with s.active():
        print(
            s.asof,
            "UST 10Y zero",
            round(s.yield_curve("UST").zeroRate(10.0, ql.Continuous).rate() * 100, 4),
        )

다음: `03_trade_to_npv.ipynb` — 계약서를 이 시장에 붙여 가격을 낸다.